# Chapter 7 — NumPy II: axes, broadcasting, linear algebra, random numbers

**Goal:** aggregate along an axis, let broadcasting do the work, reshape and stack arrays, do matrix algebra,
and, most important for this course, **simulate random experiments** with a seeded generator.

**Where you'll meet this in MLMA Lab 1:** *Linear Algebra with Numpy*, *Reshaping, Resizing and Stacking*,
*Random Numbers*, and saving/loading with `np.save` / `np.load`.

In [ ]:
import numpy as np

## 7.1 The `axis` argument

For a 2-D array, `axis=0` collapses the **rows** (one result per column) and `axis=1` collapses the
**columns** (one result per row). Think of `axis` as *"the direction that disappears"*.

```
          axis=1 →
axis=0  [[1, 2, 3],      .sum(axis=0) -> [5, 7, 9]   (shape (3,))
  ↓      [4, 5, 6]]      .sum(axis=1) -> [6, 15]     (shape (2,))
```

In a data matrix **rows are samples and columns are features**, so `X.mean(axis=0)` is the mean of each feature.

In [ ]:
A = np.array([[1, 2, 3],
              [4, 5, 6]])
print(A.sum(), A.sum(axis=0), A.sum(axis=1))
print(A.mean(axis=0), A.max(axis=1), A.argmax(axis=1))
print(np.cumsum([1, 2, 3, 4]))     # running totals (used for CDFs)

## 7.2 Broadcasting

When shapes differ, NumPy **stretches** the smaller array to fit, as long as the trailing dimensions are equal or 1.
This lets you, for example, subtract the per-column mean from every row in one line:

```
X        shape (n, d)
X.mean(axis=0)   shape (d,)    -> treated as (1, d), repeated over n rows
```

To broadcast along the other direction, add an axis: `v[:, None]` (or `v.reshape(-1, 1)`) turns shape `(n,)` into `(n, 1)`.

In [ ]:
X = np.array([[1.0, 10.0],
              [2.0, 20.0],
              [3.0, 30.0]])
print(X - X.mean(axis=0))                 # center each column
row_totals = X.sum(axis=1)
print(X / row_totals[:, None])            # each row divided by its own total

# outer product via broadcasting: (3,1) * (1,4) -> (3,4)
print(np.arange(1, 4)[:, None] * np.arange(1, 5)[None, :])

## 7.3 Reshape and stack

- `a.reshape(r, c)`: same data, new shape (sizes must match; `-1` means "work it out").
- `a.flatten()` / `a.ravel()`: back to 1-D.
- `np.vstack([a, b])` stacks rows (vertically), `np.hstack([a, b])` side by side, and
  `np.column_stack([x, y])` turns 1-D arrays into columns.
- `np.concatenate([a, b], axis=...)` is the general version.

In [ ]:
a = np.arange(6)
print(a.reshape(2, 3), a.reshape(-1, 2).shape)
r = np.array([1, 2, 3])
print(np.vstack([r, r]))
print(np.column_stack([r, r * 10]))

## 7.4 Linear algebra

| operation | code |
|---|---|
| matrix product | `A @ B` (also `np.dot(A, B)`) |
| transpose | `A.T` |
| inverse, determinant | `np.linalg.inv(A)`, `np.linalg.det(A)` |
| solve $Ax = b$ | `np.linalg.solve(A, b)` (better than `inv(A) @ b`) |
| eigen-decomposition | `vals, vecs = np.linalg.eig(A)` (`eigh` for symmetric matrices) |
| SVD | `U, s, Vt = np.linalg.svd(A)` |
| norm, trace | `np.linalg.norm(v)`, `np.trace(A)` |

**`*` vs `@`:** `A * B` multiplies element by element, `A @ B` is the matrix product.

In [ ]:
A = np.array([[2.0, 1.0],
              [1.0, 3.0]])
b = np.array([1.0, 2.0])
print(A * A)
print(A @ A)
x = np.linalg.solve(A, b)
print(x, np.allclose(A @ x, b))
vals, vecs = np.linalg.eigh(A)
print(vals)
print(np.allclose(A @ vecs[:, 0], vals[0] * vecs[:, 0]))   # A v = lambda v

## 7.5 Random numbers: use a seeded `Generator`

Modern NumPy (and this repo) uses a **generator object**:

```python
rng = np.random.default_rng(42)     # seed -> reproducible
rng.random(size)                   # uniform on [0, 1)
rng.integers(low, high, size)      # integers, high EXCLUDED: dice are rng.integers(1, 7, size)
rng.normal(loc, scale, size)       # Gaussian with mean loc, std scale
rng.binomial(n, p, size), rng.poisson(lam, size), rng.exponential(scale, size)
rng.choice(values, size, p=probs)  # sample from a finite set, optionally with given probabilities
rng.permutation(n), rng.shuffle(a)
```

The MLMA lab uses the **legacy** functions `np.random.rand(...)` and `np.random.randn(...)`.
They work, but use a hidden global state. Equivalents: `rand(2, 2)` ≈ `rng.random((2, 2))`, `randn(4)` ≈ `rng.standard_normal(4)`.

### The Monte Carlo pattern

1. Generate **all** the random draws at once, as an array with one row per experiment.
2. Compute the event per experiment with vectorized operations, giving a boolean array.
3. `.mean()` of that boolean array estimates the probability.

In [ ]:
rng = np.random.default_rng(0)
print(rng.random(3), rng.integers(1, 7, size=10), rng.normal(0, 1, size=2))

# P(sum of two dice == 7), simulated
n = 100_000
dice = rng.integers(1, 7, size=(n, 2))     # n experiments, 2 dice each
p_hat = (dice.sum(axis=1) == 7).mean()
print(p_hat, 6 / 36)

## 7.6 Saving and loading arrays

`np.save("file.npy", arr)` and `np.load("file.npy")` store a single array in NumPy's binary format.
`np.savez("file.npz", a=a, b=b)` stores several arrays by name.

---
# Exercises

### Exercise 7.1: Aggregating along axes
`sales` has one row per store and one column per month.
- `per_store`: total sales per store (shape `(3,)`)
- `per_month`: average sales per month (shape `(4,)`)
- `best_month_per_store`: for each store, the **index** of its best month (`argmax`)
- `grand_total`: the total of everything

In [ ]:
sales = np.array([[10, 12, 9, 14],
                  [20, 18, 25, 22],
                  [5, 7, 6, 4]])
per_store = ...
per_month = ...
best_month_per_store = ...
grand_total = ...

In [ ]:
assert np.array_equal(per_store, [45, 85, 22])
assert np.allclose(per_month, [35 / 3, 37 / 3, 40 / 3, 40 / 3])
assert np.array_equal(best_month_per_store, [3, 2, 1])
assert grand_total == 152
print("Exercise 7.1 passed")

### Exercise 7.2: Standardize features with broadcasting
Standardize each **column** of `X`: subtract the column mean and divide by the column standard deviation.
Store the result in `X_std`. One line, no loops.

In [ ]:
X = np.array([[22.0, 7.25],
              [38.0, 71.28],
              [26.0, 7.92],
              [35.0, 53.10]])
X_std = ...

In [ ]:
assert X_std.shape == X.shape
assert np.allclose(X_std.mean(axis=0), 0) and np.allclose(X_std.std(axis=0), 1)
print("Exercise 7.2 passed")

### Exercise 7.3: Normalize rows into probability distributions
Each row of `counts` holds counts over 3 categories. Turn every row into a probability distribution
(divide by its own row total) and store it in `P`. You'll need `[:, None]` or `keepdims=True`.

In [ ]:
counts = np.array([[2, 3, 5],
                   [10, 0, 10],
                   [1, 1, 2]])
P = ...

In [ ]:
assert np.allclose(P.sum(axis=1), 1)
assert np.allclose(P[0], [0.2, 0.3, 0.5]) and np.allclose(P[2], [0.25, 0.25, 0.5])
print("Exercise 7.3 passed")

### Exercise 7.4: Reshape and stack
- `grid`: `np.arange(1, 13)` reshaped to 3 rows and 4 columns
- `flat`: `grid` flattened back to 1-D
- `design`: a matrix whose first column is all ones and second column is `x`.
  This is the design matrix for linear regression; use `np.column_stack` and `np.ones_like`.
- `both`: `top` stacked **above** `bottom` (shape `(2, 3)`)

In [ ]:
x = np.array([0.5, 1.5, 2.5])
top = np.array([1, 2, 3])
bottom = np.array([4, 5, 6])
grid = ...
flat = ...
design = ...
both = ...

In [ ]:
assert grid.shape == (3, 4) and grid[2, 3] == 12
assert np.array_equal(flat, np.arange(1, 13))
assert np.array_equal(design, [[1, 0.5], [1, 1.5], [1, 2.5]])
assert np.array_equal(both, [[1, 2, 3], [4, 5, 6]])
print("Exercise 7.4 passed")

### Exercise 7.5: `*` vs `@`
**Predict first**: for `A = [[1, 2], [3, 4]]` and `B = [[0, 1], [1, 0]]`, write the result matrices as nested lists:
- `pred_elementwise`: `A * B`
- `pred_matmul`: `A @ B`

In [ ]:
A = np.array([[1, 2], [3, 4]])
B = np.array([[0, 1], [1, 0]])
pred_elementwise = ...
pred_matmul = ...

In [ ]:
assert np.array_equal(pred_elementwise, A * B), f"A * B is\n{A * B}"
assert np.array_equal(pred_matmul, A @ B), f"A @ B is\n{A @ B}"
print("Exercise 7.5 passed")

### Exercise 7.6: Solve a linear system
Solve
$$\begin{aligned} 3x + y - z &= 1 \\ x - 2y + 4z &= 11 \\ 2x + y + z &= 6 \end{aligned}$$
Build the coefficient matrix `A` and right-hand side `b`, then `sol = np.linalg.solve(A, b)`.
Also compute `det_A`.

In [ ]:
A = ...
b = ...
sol = ...
det_A = ...

In [ ]:
assert np.allclose(A @ sol, b), "A @ sol should equal b"
assert np.allclose(sol, [1, 1, 3]) and np.isclose(det_A, -16)
print("Exercise 7.6 passed")

### Exercise 7.7: Eigenvalues of a covariance matrix
`C` is a symmetric covariance matrix. Use `np.linalg.eigh` (eigenvalues come out in **ascending** order):
- `vals`, `vecs` = the eigen-decomposition
- `top_vec`: the eigenvector belonging to the **largest** eigenvalue (it's a *column* of `vecs`)
- `explained`: the fraction of total variance along `top_vec`: largest eigenvalue / sum of eigenvalues
  (the idea behind PCA)

In [ ]:
C = np.array([[4.0, 2.0],
              [2.0, 3.0]])
vals, vecs = ..., ...
top_vec = ...
explained = ...

In [ ]:
assert np.allclose(C @ top_vec, vals.max() * top_vec), "top_vec should satisfy C v = lambda_max v"
assert np.isclose(explained, vals.max() / np.trace(C))
print("Exercise 7.7 passed")

### Exercise 7.8: Random draws with a generator
With `rng = np.random.default_rng(123)` generate, **in this order**:
- `u`: 5 uniform numbers in [0, 1)
- `dice`: 10 rolls of a fair die (values 1..6, so remember `high` is excluded)
- `heights`: 1000 normal draws with mean 175 and standard deviation 7
- `colors`: 8 draws from `["red", "green", "blue"]` with probabilities `[0.5, 0.3, 0.2]`

In [ ]:
rng = np.random.default_rng(123)
u = ...
dice = ...
heights = ...
colors = ...

In [ ]:
_r = np.random.default_rng(123)
assert np.allclose(u, _r.random(5)), "u: same seed and order should give the same numbers"
assert np.array_equal(dice, _r.integers(1, 7, size=10)) and dice.min() >= 1 and dice.max() <= 6
assert np.allclose(heights, _r.normal(175, 7, size=1000))
assert colors.shape == (8,) and set(colors) <= {"red", "green", "blue"}
print("Exercise 7.8 passed")

### Exercise 7.9: Monte Carlo: the birthday problem
**Predict first:** in a room of 23 people, what's the probability that at least two share a birthday (365 equally likely days)?
Write it in `my_guess`.

Then simulate `n_sim = 50_000` rooms **without a Python loop over rooms**:
1. `bdays = rng.integers(0, 365, size=(n_sim, 23))`
2. sort each row (`np.sort(bdays, axis=1)`) and check whether any two **neighbours** are equal:
   `np.diff(sorted_rows, axis=1) == 0`, then `.any(axis=1)`
3. `p_hat` = the mean of that boolean array

In [ ]:
my_guess = ...
rng = np.random.default_rng(2026)
n_sim = 50_000
pass  # YOUR CODE HERE
print(f"guess {my_guess}, simulated {p_hat:.4f}")

In [ ]:
exact = 1 - np.prod((365 - np.arange(23)) / 365)
assert abs(p_hat - exact) < 0.01, f"simulated {p_hat}, exact is {exact:.4f}"
print(f"Exercise 7.9 passed (exact: {exact:.4f}). Surprised?")

### Exercise 7.10: Monte Carlo: estimate π
Draw `n = 200_000` points uniformly in the square $[-1, 1]^2$ (`rng.uniform(-1, 1, size=(n, 2))`).
The fraction landing inside the unit circle estimates $\pi/4$. Store the estimate of π in `pi_hat`.

Bonus: what happens to the error when you use 100× more points?

In [ ]:
rng = np.random.default_rng(0)
n = 200_000
pi_hat = ...
print(pi_hat)

In [ ]:
assert abs(pi_hat - np.pi) < 0.02, f"got {pi_hat}"
print("Exercise 7.10 passed")

### Exercise 7.11: Law of large numbers: running mean
Flip a biased coin (`p = 0.3`) `n = 5000` times as an array of 0/1 values (`rng.binomial(1, p, size=n)`, or a uniform and a comparison).
Compute `running_mean`, where element `k` is the average of the first `k + 1` flips.
Use `np.cumsum` and `np.arange`. No loops.

In [ ]:
rng = np.random.default_rng(1)
p, n = 0.3, 5000
flips = ...
running_mean = ...

In [ ]:
assert running_mean.shape == (n,)
assert running_mean[0] in (0, 1) and np.isclose(running_mean[-1], flips.mean())
assert abs(running_mean[-1] - p) < 0.03
print("Exercise 7.11 passed. You'll plot this in Chapter 8")

### Exercise 7.12: Save and load
Save `results` to `"primer_results.npy"`, load it back into `loaded`, then delete the file
(`Path("primer_results.npy").unlink()`).

In [ ]:
from pathlib import Path
results = np.random.default_rng(5).normal(size=(3, 3))
loaded = ...

In [ ]:
assert np.array_equal(loaded, results)
assert not Path("primer_results.npy").exists()
print("Exercise 7.12 passed")

---
**Chapter 7 done.** You can now read the whole NumPy part of the MLMA lab. Next: Chapter 8, plotting with Matplotlib.